## Linear Classifier in TensorFlow 
Using Low Level API in Eager Execution mode

In [97]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [0]:
import pandas as pd


In [0]:
data = pd.read_csv('/content/drive/My Drive/Lab/Iris.csv')

In [4]:
data.head()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


### Load tensorflow

In [0]:
import tensorflow as tf

### Collect Data

In [0]:
from google.colab import drive
drive.mount('/gdrive')

In [0]:
import pandas as pd

In [0]:
data_prices = pd.read_csv('/content/drive/My Drive/Lab/prices.csv')

### Check all columns in the dataset

In [25]:
data_prices.columns

Index(['date', 'symbol', 'open', 'close', 'low', 'high', 'volume'], dtype='object')

### Drop columns `date` and  `symbol`

In [0]:
data_prices = data_prices.drop(['date','symbol'], axis=1)

In [27]:
data_prices.head()

,open,close,low,high,volume
0,123.430000,125.839996,122.309998,126.250000,2163600.0
1,125.239998,119.980003,119.940002,125.540001,2386400.0
2,116.379997,114.949997,114.930000,119.739998,2489500.0
3,115.480003,116.620003,113.500000,117.440002,2006300.0
4,117.010002,114.970001,114.089996,117.330002,1408600.0


### Consider only first 1000 rows in the dataset for building feature set and target set
Target 'Volume' has very high values. Divide 'Volume' by 1000,000

In [28]:
data_prices.shape

(851264, 5)

In [0]:
data_price1k = data_prices.head(1000)

In [33]:
data_price1k.shape

(1000, 5)

In [34]:
 data_price1k['volume'] = data_price1k['volume'] / 1000000

/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  """Entry point for launching an IPython kernel.


In [35]:
data_price1k.head()

,open,close,low,high,volume
0,123.430000,125.839996,122.309998,126.250000,2.1636
1,125.239998,119.980003,119.940002,125.540001,2.3864
2,116.379997,114.949997,114.930000,119.739998,2.4895
3,115.480003,116.620003,113.500000,117.440002,2.0063
4,117.010002,114.970001,114.089996,117.330002,1.4086


### Divide the data into train and test sets

In [0]:
from sklearn.model_selection import train_test_split

In [0]:
X = data_price1k.drop('volume', axis=1)
y = data_price1k['volume']

In [0]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)

#### Convert Training and Test Data to numpy float32 arrays


In [40]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 700 entries, 731 to 37
Data columns (total 4 columns):
open     700 non-null float64
close    700 non-null float64
low      700 non-null float64
high     700 non-null float64
dtypes: float64(4)
memory usage: 27.3 KB


In [0]:
X_train = X_train.to_numpy('float32')
X_test =  X_test.to_numpy('float32')
y_train = y_train.to_numpy('float32')
y_test = y_test.to_numpy('float32')

In [52]:
type(X_test)

numpy.ndarray

### Normalize the data
You can use Normalizer from sklearn.preprocessing

In [0]:
from sklearn.preprocessing import Normalizer

#x_train_n = normalize(np.vstack([X_train]), norm='l2', axis=1)
#x_test_n = normalize(np.vstack([X_test]), norm='l2', axis=1)


In [0]:
transformer = Normalizer().fit(X_train)
X_train_new = transformer.transform(X_train)

In [0]:
transformer = Normalizer().fit(X_test)
X_test_new = transformer.transform(X_test)

## Building the Model in tensorflow

1.Define Weights and Bias, use tf.zeros to initialize weights and Bias

In [0]:
W = tf.Variable(tf.random_normal(shape=[4,1]), name="Weights")
b = tf.Variable(tf.zeros(shape=[1]),name="Bias")

In [94]:
W

<tf.Variable 'Weights_3:0' shape=(4, 1) dtype=float32_ref>

2.Define a function to calculate prediction

In [0]:
y = tf.add(tf.matmul(X_train_new,W),b,name='output')

3.Loss (Cost) Function [Mean square error]

In [0]:
loss = tf.reduce_mean(tf.square(y-y_train),name='mse')

4.Function to train the Model

1.   Record all the mathematical steps to calculate Loss
2.   Calculate Gradients of Loss w.r.t weights and bias
3.   Update Weights and Bias based on gradients and learning rate to minimize loss

In [0]:
train_op = tf.train.GradientDescentOptimizer(0.03).minimize(loss)

## Train the model for 100 epochs 
1. Observe the training loss at every iteration
2. Observe Train loss at every 5th iteration

In [0]:
#Input features
x = tf.placeholder(shape=[None,4],dtype=tf.float32, name='x-input')

#Normalize the data
x_n = tf.nn.l2_normalize(x,1)

#Actual Prices
y_ = tf.placeholder(shape=[None],dtype=tf.float32, name='y-input')

In [0]:
#Lets start graph Execution
sess = tf.Session()

# variables need to be initialized before we can use them
sess.run(tf.global_variables_initializer())

#how many times data need to be shown to model
training_epochs = 100

In [117]:
for epoch in range(training_epochs):
            
    #Calculate train_op and loss
    _, train_loss = sess.run([train_op,loss],feed_dict={x:X_train_new, y_:y_train})
    
    print ('Training loss at step: ', epoch, ' is ', train_loss)

Training loss at step:  0  is  225.87958
Training loss at step:  1  is  219.00435
Training loss at step:  2  is  213.68013
Training loss at step:  3  is  209.55702
Training loss at step:  4  is  206.36406
Training loss at step:  5  is  203.89143
Training loss at step:  6  is  201.97662
Training loss at step:  7  is  200.49374
Training loss at step:  8  is  199.34541
Training loss at step:  9  is  198.45616
Training loss at step:  10  is  197.76749
Training loss at step:  11  is  197.23419
Training loss at step:  12  is  196.82121
Training loss at step:  13  is  196.50139
Training loss at step:  14  is  196.25371
Training loss at step:  15  is  196.0619
Training loss at step:  16  is  195.9134
Training loss at step:  17  is  195.79837
Training loss at step:  18  is  195.70929
Training loss at step:  19  is  195.6403
Training loss at step:  20  is  195.58688
Training loss at step:  21  is  195.54553
Training loss at step:  22  is  195.51349
Training loss at step:  23  is  195.4887
Traini

In [118]:
print(sess.run(W))

[[-0.72241455]
 [-0.05216902]
 [-0.6659141 ]
 [ 0.5146053 ]]


In [0]:
sess.close()

In [0]:
#Lets start graph Execution
sess = tf.Session()

# variables need to be initialized before we can use them
sess.run(tf.global_variables_initializer())

#how many times data need to be shown to model
training_epochs = 100

In [121]:
#print training loss for every 5th iteration
for epoch in range(training_epochs):
            
    #Calculate train_op and loss
    _, train_loss = sess.run([train_op,loss],feed_dict={x:X_train_new, y_:y_train})
    
    if epoch % 5 == 0:
      print ('Training loss at step: ', epoch, ' is ', train_loss)

Training loss at step:  0  is  225.87958
Training loss at step:  5  is  203.89143
Training loss at step:  10  is  197.76749
Training loss at step:  15  is  196.0619
Training loss at step:  20  is  195.58688
Training loss at step:  25  is  195.45459
Training loss at step:  30  is  195.41774
Training loss at step:  35  is  195.4075
Training loss at step:  40  is  195.40462
Training loss at step:  45  is  195.40382
Training loss at step:  50  is  195.4036
Training loss at step:  55  is  195.40355
Training loss at step:  60  is  195.40353
Training loss at step:  65  is  195.40355
Training loss at step:  70  is  195.40353
Training loss at step:  75  is  195.40353
Training loss at step:  80  is  195.40353
Training loss at step:  85  is  195.40353
Training loss at step:  90  is  195.40355
Training loss at step:  95  is  195.40353


In [122]:
print(sess.run(W))

[[-1.0801225 ]
 [-0.30398908]
 [-0.6301103 ]
 [-0.12995513]]


In [123]:
print(sess.run(b))

[0.]


In [0]:
sess.close()

### Get the shapes and values of W and b

In [107]:
W

<tf.Variable 'Weights_3:0' shape=(4, 1) dtype=float32_ref>

In [108]:
b

<tf.Variable 'Bias_3:0' shape=(1,) dtype=float32_ref>

### Model Prediction on 1st Examples in Test Dataset

In [0]:
y_test_pred = tf.add(tf.matmul(X_test_new,W),b,name='output')

## Classification using tf.Keras

In this exercise, we will build a Deep Neural Network using tf.Keras. We will use Iris Dataset for this exercise.

### Load the given Iris data using pandas (Iris.csv)

In [0]:
data = pd.read_csv('/content/drive/My Drive/Lab/Iris.csv')

In [0]:
data = data.drop('Id',axis=1)

### Target set has different categories. So, Label encode them. And convert into one-hot vectors using get_dummies in pandas.

In [195]:
data.head()

,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [0]:
X = data.iloc[:,0:4].values
y = data.iloc[:,4].values

from sklearn.preprocessing import LabelEncoder
encoder =  LabelEncoder()
y1 = encoder.fit_transform(y)

Y = pd.get_dummies(y1).values

#from sklearn.model_selection import train_test_split
#X_train,X_test, y_train,y_test = train_test_split(X,y1,test_size=0.2,random_state=0)

### Splitting the data into feature set and target set

In [0]:
#done above

###  Building Model in tf.keras

Build a Linear Classifier model  <br>
1.  Use Dense Layer  with input shape of 4 (according to the feature set) and number of outputs set to 3<br> 
2. Apply Softmax on Dense Layer outputs <br>
3. Use SGD as Optimizer
4. Use categorical_crossentropy as loss function 

In [0]:
from keras.models import Sequential
from keras.layers import Dense
from keras.optimizers import SGD


In [0]:
model = Sequential()
model.add(Dense(12, input_dim=4, activation='relu'))
model.add(Dense(3, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='sgd', metrics=['accuracy'])

### Model Training 

In [200]:
model.fit(X,Y,epochs=100,batch_size=10)

Epoch 1/100
150/150 [==============================] - 0s 3ms/step - loss: 1.5018 - acc: 0.3333
Epoch 2/100
150/150 [==============================] - 0s 281us/step - loss: 1.1628 - acc: 0.3333
Epoch 3/100
150/150 [==============================] - 0s 276us/step - loss: 1.0875 - acc: 0.3333
Epoch 4/100
150/150 [==============================] - 0s 287us/step - loss: 1.0363 - acc: 0.3333
Epoch 5/100
150/150 [==============================] - 0s 235us/step - loss: 0.9974 - acc: 0.3333
Epoch 6/100
150/150 [==============================] - 0s 259us/step - loss: 0.9627 - acc: 0.3333
Epoch 7/100
150/150 [==============================] - 0s 326us/step - loss: 0.9302 - acc: 0.3333
Epoch 8/100
150/150 [==============================] - 0s 255us/step - loss: 0.9015 - acc: 0.3867
Epoch 9/100
150/150 [==============================] - 0s 245us/step - loss: 0.8761 - acc: 0.6000
Epoch 10/100
150/150 [==============================] - 0s 260us/step - loss: 0.8506 - acc: 0.6600
Epoch 11/100
150/150 

### Model Prediction

In [201]:
model.evaluate(X,Y)

150/150 [==============================] - 0s 448us/step


[0.23126144955555597, 0.9]

### Save the Model

In [0]:
import pickle
pickle.dump(model, open("model.pkl","wb" , ))
#loading a model from a file called model.pkl
model = pickle.load(open("model.pkl", "rb"))
##"rb" : this means read mode

### Build and Train a Deep Neural network with 2 hidden layer  - Optional - For Practice

Does it perform better than Linear Classifier? What could be the reason for difference in performance?